In [0]:
import requests
import numpy as np
import pandas as pd
 
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lpad, hour, first, row_number, count as F_count, sum as F_sum, avg as F_avg, countDistinct
from pyspark.sql.types import IntegerType, StringType
from pyspark.sql.window import Window
 

In [0]:
store       = spark.table("workspace.cstore_sample.cstore_stores")
ctin             = spark.table("workspace.cstore_sample.cstore_master_ctin")
discounts        = spark.table("workspace.cstore_sample.cstore_discounts")
transaction_sets = spark.table("workspace.cstore_sample.cstore_transaction_sets")
payments         = spark.table("workspace.cstore_sample.cstore_payments")
items = spark.table("workspace.cstore_sample.cstore_transaction_items")
 
discounts1 = discounts.drop("DISCOUNT_NUMBER")
 
#items = items.filter(~F.col("NONSCAN_CATEGORY").rlike(r"^\d+\.\d+$"))

In [0]:
items = items.withColumn("HOUR", hour("DATE_TIME")).withColumn(
    "daypart",
    F.when((F.col("HOUR") >= 5) & (F.col("HOUR") < 11), "morning")
     .when((F.col("HOUR") >= 11) & (F.col("HOUR") < 17), "afternoon")
     .when((F.col("HOUR") >= 17) & (F.col("HOUR") < 22), "evening")
     .otherwise("night")
)
#print(items.show())
#items.select("daypart").show()
items.select("STORE_ID","daypart").count()

In [0]:
print("Distinct stores:", items.select("STORE_ID").distinct().count())
print("Distinct dayparts:", items.select("daypart").count())

In [0]:
APIKEY = "08807ea115bb851c24144cae7a255a4ae7f82208"
 
# basic variables: income, age, population, commute-time breakdown
zcta_vars = {
    "B19013_001E": "median_hh_income", "B01002_001E": "median_age",
    "B01003_001E": "total_pop", "B19301_001E": "per_capita_income",
    "B11001_001E": "total_households",
    "B08303_001E": "total_commuters", "B08303_002E": "commute_lt_5", "B08303_003E": "commute_5_9",
    "B08303_004E": "commute_10_14", "B08303_005E": "commute_15_19", "B08303_006E": "commute_20_24",
    "B08303_007E": "commute_25_29", "B08303_008E": "commute_30_34", "B08303_009E": "commute_35_39",
    "B08303_010E": "commute_40_44", "B08303_011E": "commute_45_59", "B08303_012E": "commute_60_89",
    "B08303_013E": "commute_90plus",
}
resp = requests.get(
    "https://api.census.gov/data/2023/acs/acs5",
    params={"get": ",".join(zcta_vars.keys()), "for": "zip code tabulation area:*", "key": APIKEY},
)
rows = resp.json()
acs_basic = pd.DataFrame(rows[1:], columns=rows[0]).rename(columns={**zcta_vars, "zip code tabulation area": "ZCTA"})
 

In [0]:
# --- pull poverty_pct from the profile endpoint ---
profile_vars = {"DP03_0119PE": "poverty_pct"}
resp2 = requests.get(
    "https://api.census.gov/data/2023/acs/acs5/profile",
    params={"get": "NAME," + ",".join(profile_vars), "for": "zip code tabulation area:*", "key": APIKEY},
)
rows2 = resp2.json()
acs_profile = pd.DataFrame(rows2[1:], columns=rows2[0]).rename(columns={**profile_vars, "zip code tabulation area": "ZCTA"})

num_cols_basic = list(zcta_vars.values())
acs_basic[num_cols_basic] = acs_basic[num_cols_basic].apply(pd.to_numeric, errors="coerce").mask(lambda d: d < 0)
acs_profile["poverty_pct"] = pd.to_numeric(acs_profile["poverty_pct"], errors="coerce")
acs_profile.loc[acs_profile["poverty_pct"] < 0, "poverty_pct"] = np.nan

# --- merge basic + poverty into one demographics table ---
acs = acs_basic.merge(acs_profile[["ZCTA", "poverty_pct"]], on="ZCTA", how="left")
acs["ZCTA"] = acs["ZCTA"].astype(str).str.zfill(5)

# --- compute avg commute time from the bucket breakdown ---
commute_cols = ["commute_lt_5", "commute_5_9", "commute_10_14", "commute_15_19", "commute_20_24",
                 "commute_25_29", "commute_30_34", "commute_35_39", "commute_40_44", "commute_45_59",
                 "commute_60_89", "commute_90plus"]
midpoints = [2.5, 7, 12, 17, 22, 27, 32, 37, 42, 52, 74.5, 105]
acs["avg_commute_minutes"] = sum(acs[c] * m for c, m in zip(commute_cols, midpoints)) / acs["total_commuters"].replace(0, np.nan)

# --- merge demographics onto stores ---
stores_pd = store.toPandas()
stores_pd["ZIP5"] = stores_pd["ZIP_CODE"].astype(str).str.zfill(5)
stores_pd["STORE_ID"] = stores_pd["STORE_ID"].astype(str)

demo_cols = ["median_hh_income", "median_age", "total_pop", "per_capita_income",
             "total_households", "total_commuters", "avg_commute_minutes", "poverty_pct"]
stores_pd = stores_pd.merge(acs[["ZCTA"] + demo_cols], left_on="ZIP5", right_on="ZCTA", how="left")
stores_pd = stores_pd[stores_pd["total_pop"] > 0].copy()  # drop the store with no matching ZCTA data

print(f"{len(stores_pd)} stores with complete demographic data")

# --- now build the readable per-store demographic profile ---
profile_feats = ["median_hh_income", "poverty_pct", "avg_commute_minutes", "total_pop"]

def bin_tier(series, labels=("Low", "Medium", "High")):
    return pd.qcut(series, q=3, labels=labels, duplicates="drop")

for feat in profile_feats:
    stores_pd[f"{feat}_tier"] = bin_tier(stores_pd[feat])

stores_pd["profile"] = (
    stores_pd["median_hh_income_tier"].astype(str) + " income, "
    + stores_pd["poverty_pct_tier"].astype(str) + " poverty, "
    + stores_pd["avg_commute_minutes_tier"].astype(str) + " commute, "
    + stores_pd["total_pop_tier"].astype(str) + " population"
)

print(stores_pd[["STORE_ID", "profile"] + profile_feats].head(15))

In [0]:
store_rollup = (
    items.groupBy("STORE_ID")
    .agg(
        F.sum("GRAND_TOTAL_AMOUNT").alias("total_revenue"),
        F.avg("GRAND_TOTAL_AMOUNT").alias("avg_basket_size"),
        F.countDistinct("TRANSACTION_SET_ID").alias("total_transactions"),
        F.countDistinct("NONSCAN_CATEGORY").alias("distinct_categories"),
    )
    .toPandas()
)
store_rollup["STORE_ID"] = store_rollup["STORE_ID"].astype(str)
 
feature_table = stores_pd.merge(store_rollup, on="STORE_ID", how="left")
feature_table = feature_table[feature_table["total_pop"] > 0].copy()  # drop the 1 store with no ZCTA population data
print(f"{len(feature_table)} stores with complete features")

In [0]:
fuel_txn_ids = (
    items.filter("NONSCAN_CATEGORY = 'FUEL' AND GRAND_TOTAL_AMOUNT > 0")
    .select("STORE_ID", "TRANSACTION_SET_ID").distinct()
)
instore_txn_ids = (
    items.filter(
        F.col("SCAN_TYPE").isin("GTIN", "PLU", "FMT_ERR") |
        F.col("NONSCAN_CATEGORY").isin("MERCHANDISE", "FOODSERVICE", "CARDS", "LOTTERY", "SERVICES", "LICENSE", "MONEY ORDER")
    ).select("TRANSACTION_SET_ID").distinct().withColumn("converted", F.lit(1))
)
conversion = (
    fuel_txn_ids.join(instore_txn_ids, on="TRANSACTION_SET_ID", how="left")
    .withColumn("converted", F.coalesce(F.col("converted"), F.lit(0)))
    .groupBy("STORE_ID")
    .agg(F.avg("converted").alias("inside_conversion_rate"))
    .toPandas()
)
conversion["STORE_ID"] = conversion["STORE_ID"].astype(str)

feature_table = feature_table.merge(conversion, on="STORE_ID", how="left")
feature_table["inside_conversion_rate"] = feature_table["inside_conversion_rate"].fillna(0)

print(feature_table["inside_conversion_rate"].describe())

In [0]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_feats = ["median_hh_income", "per_capita_income", "poverty_pct", "avg_commute_minutes", "total_pop"]

feature_table[cluster_feats] = feature_table[cluster_feats].fillna(feature_table[cluster_feats].median())

scaler = StandardScaler().fit(feature_table[cluster_feats])
X_scaled = scaler.transform(feature_table[cluster_feats])

k = 5
km = KMeans(n_clusters=k, random_state=42, n_init=10)
feature_table["segment"] = km.fit_predict(X_scaled)

print(feature_table["segment"].value_counts().sort_index())
print("Silhouette score:", round(silhouette_score(X_scaled, feature_table["segment"]), 4))

In [0]:
print(feature_table[cluster_feats].skew().sort_values(ascending=False))

In [0]:
skewed_feats = ["median_hh_income", "per_capita_income", "poverty_pct"]

def flag_outliers_iqr(df, col):
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (df[col] < lower) | (df[col] > upper)

outlier_flags = pd.DataFrame({col: flag_outliers_iqr(feature_table, col) for col in skewed_feats})
feature_table["is_extreme"] = outlier_flags.any(axis=1)

print(feature_table["is_extreme"].value_counts())

In [0]:
# typical stores
typical = feature_table[~feature_table["is_extreme"]].copy()
scaler_typical = StandardScaler().fit(typical[cluster_feats])
X_typical = scaler_typical.transform(typical[cluster_feats])

km_typical = KMeans(n_clusters=5, random_state=42, n_init=10)
typical["segment"] = km_typical.fit_predict(X_typical)
print("Typical segments:")
print(typical["segment"].value_counts().sort_index())
print("Silhouette:", round(silhouette_score(X_typical, typical["segment"]), 4))

# extreme stores
extreme = feature_table[feature_table["is_extreme"]].copy()
scaler_extreme = StandardScaler().fit(extreme[cluster_feats])
X_extreme = scaler_extreme.transform(extreme[cluster_feats])

km_extreme = KMeans(n_clusters=2, random_state=42, n_init=10)
extreme["segment"] = km_extreme.fit_predict(X_extreme)
print("\nExtreme segments:")
print(extreme["segment"].value_counts().sort_index())
print("Silhouette:", round(silhouette_score(X_extreme, extreme["segment"]), 4))

In [0]:
typical["final_segment"] = "typical_" + typical["segment"].astype(str)
extreme["final_segment"] = "extreme_" + extreme["segment"].astype(str)

feature_table_final = pd.concat([typical, extreme], ignore_index=True)

print(feature_table_final["final_segment"].value_counts().sort_index())
print(f"\nTotal stores: {len(feature_table_final)}")

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, classification_report

tree_feats = ["median_hh_income", "per_capita_income", "poverty_pct", "avg_commute_minutes", "total_pop",
              "total_revenue", "avg_basket_size", "total_transactions", "distinct_categories", "inside_conversion_rate"]

X_demo = feature_table_final[tree_feats].fillna(0)
y_seg = feature_table_final["final_segment"]

X_train, X_test, y_train, y_test = train_test_split(X_demo, y_seg, test_size=0.2, random_state=42, stratify=y_seg)

tree = DecisionTreeClassifier(max_depth=5, min_samples_leaf=8, random_state=42)
tree.fit(X_train, y_train)

test_pred = tree.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Majority-class baseline:", round(y_seg.value_counts(normalize=True).max(), 4))
print()
print(classification_report(y_test, test_pred, zero_division=0))

In [0]:
restricted_subcategories = ["CIGARETTES", "TOBACCO", "OTP", "ALCOHOL","LOTTERY"]
segment_lookup = feature_table_final[["STORE_ID", "final_segment"]].copy()
segment_lookup["STORE_ID"] = segment_lookup["STORE_ID"].astype(str)

segment_daypart_sales = (
    items.filter(F.col("NONSCAN_CATEGORY") != "FUEL")
    .filter(~F.col("NONSCAN_SUBCATEGORY").isin(restricted_subcategories))
    .filter(F.col("NONSCAN_SUBCATEGORY").isNotNull())
    .groupBy("STORE_ID", "daypart", "NONSCAN_SUBCATEGORY")
    .agg(F.sum("UNIT_QUANTITY").alias("units_sold"))
    .toPandas()
)
segment_daypart_sales["STORE_ID"] = segment_daypart_sales["STORE_ID"].astype(str)
segment_daypart_sales = segment_daypart_sales.merge(segment_lookup, on="STORE_ID", how="inner")

segment_profile = (
    segment_daypart_sales.groupby(["final_segment", "daypart", "NONSCAN_SUBCATEGORY"])["units_sold"]
    .sum().reset_index()
)

def top_products(segment_id, daypart, n=3):
    sub = segment_profile[(segment_profile["final_segment"] == segment_id) & (segment_profile["daypart"] == daypart)]
    return sub.sort_values("units_sold", ascending=False).head(n)[["NONSCAN_SUBCATEGORY", "units_sold"]]

for s in sorted(feature_table_final["final_segment"].unique()):
    print(f"\n{s} — top products by daypart:")
    for dp in ["morning", "afternoon", "evening", "night"]:
        top3 = top_products(s, dp)["NONSCAN_SUBCATEGORY"].tolist()
        print(f"  {dp}: {top3}")

In [0]:
ctin = spark.table("workspace.cstore_sample.cstore_master_ctin")
#ctin.select("BRAND", "SKUPOS_DESCRIPTION").distinct().show(30, truncate=False)

In [0]:
items_scanned = items.filter(F.col("SCAN_TYPE") == "GTIN")

joined = items_scanned.join(
    ctin.select("GTIN", "CATEGORY", "SUBCATEGORY", "BRAND", "SKUPOS_DESCRIPTION"),
    on="GTIN", how="inner"
)
print("Rows after join:", joined.count())
joined.select("CATEGORY", "BRAND", "SKUPOS_DESCRIPTION").show(10, truncate=False)

In [0]:
# paste the exact category names from the check above into this list
restricted_categories = ["Cigarettes", "Other Tobacco Products", "Beer", "Lottery"]  # + whatever Fireball/vodka showed

segment_daypart_brand = (
    items_scanned
    .join(ctin.select("GTIN", "CATEGORY", "BRAND", "SKUPOS_DESCRIPTION"), on="GTIN", how="inner")
    .filter(~F.col("CATEGORY").isin(restricted_categories))
    .filter(F.col("SKUPOS_DESCRIPTION").isNotNull())
    .groupBy("STORE_ID", "daypart", "BRAND", "SKUPOS_DESCRIPTION")
    .agg(F.sum("UNIT_QUANTITY").alias("units_sold"))
    .toPandas()
)
segment_daypart_brand["STORE_ID"] = segment_daypart_brand["STORE_ID"].astype(str)
segment_daypart_brand = segment_daypart_brand.merge(segment_lookup, on="STORE_ID", how="inner")
segment_daypart_brand["item_label"] = segment_daypart_brand["BRAND"].fillna("") + " - " + segment_daypart_brand["SKUPOS_DESCRIPTION"]

segment_brand_profile = (
    segment_daypart_brand.groupby(["final_segment", "daypart", "item_label"])["units_sold"]
    .sum().reset_index()
)

for s in sorted(feature_table_final["final_segment"].unique()):
    print(f"\n{s} — top branded items by daypart:")
    for dp in ["morning", "afternoon", "evening", "night"]:
        sub = segment_brand_profile[(segment_brand_profile["final_segment"] == s) & (segment_brand_profile["daypart"] == dp)]
        print(f"  {dp}: {sub.sort_values('units_sold', ascending=False).head(5)['item_label'].tolist()}")

In [0]:
restricted_categories = ["Cigarettes", "Other Tobacco Products", "Beer", "Liquor", "Lottery/Gaming"]

segment_daypart_brand = (
    items_scanned
    .join(ctin.select("GTIN", "CATEGORY", "BRAND", "SKUPOS_DESCRIPTION"), on="GTIN", how="inner")
    .filter(~F.col("CATEGORY").isin(restricted_categories))
    .filter(F.col("SKUPOS_DESCRIPTION").isNotNull())
    .groupBy("STORE_ID", "daypart", "BRAND", "SKUPOS_DESCRIPTION")
    .agg(F.sum("UNIT_QUANTITY").alias("units_sold"))
    .toPandas()
)
segment_daypart_brand["STORE_ID"] = segment_daypart_brand["STORE_ID"].astype(str)
segment_daypart_brand["item_label"] = segment_daypart_brand["BRAND"].fillna("") + " - " + segment_daypart_brand["SKUPOS_DESCRIPTION"]
segment_daypart_brand = segment_daypart_brand.merge(segment_lookup, on="STORE_ID", how="inner")

segment_brand_profile = (
    segment_daypart_brand.groupby(["final_segment", "daypart", "item_label"])["units_sold"]
    .sum().reset_index()
)

def top_branded_items(segment_id, daypart, n=5):
    sub = segment_brand_profile[(segment_brand_profile["final_segment"] == segment_id) & (segment_brand_profile["daypart"] == daypart)]
    return sub.sort_values("units_sold", ascending=False).head(n)[["item_label", "units_sold"]]

for s in sorted(feature_table_final["final_segment"].unique()):
    print(f"\n{s} — top branded items by daypart:")
    for dp in ["morning", "afternoon", "evening", "night"]:
        print(f"  {dp}: {top_branded_items(s, dp)['item_label'].tolist()}")

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

tree_feats = ["median_hh_income", "per_capita_income", "poverty_pct", "avg_commute_minutes", "total_pop",
              "total_revenue", "avg_basket_size", "total_transactions", "distinct_categories",
               "inside_conversion_rate"]

X_demo = feature_table_final[tree_feats].fillna(0)
y_seg = feature_table_final["final_segment"]

X_train, X_test, y_train, y_test = train_test_split(X_demo, y_seg, test_size=0.2, random_state=42, stratify=y_seg)

tree = DecisionTreeClassifier(max_depth=5, min_samples_leaf=8, random_state=42)
tree.fit(X_train, y_train)

test_pred = tree.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Majority-class baseline:", round(y_seg.value_counts(normalize=True).max(), 4))
print(classification_report(y_test, test_pred, zero_division=0))

In [0]:
def recommend_item_for_store(store_id: str, hour_of_day: int, n=3):
    row = feature_table_final[feature_table_final["STORE_ID"] == str(store_id)]
    if row.empty:
        return None
    x = row[tree_feats].fillna(0)
    seg = tree.predict(x)[0]

    if 5 <= hour_of_day < 11: daypart = "morning"
    elif 11 <= hour_of_day < 17: daypart = "afternoon"
    elif 17 <= hour_of_day < 22: daypart = "evening"
    else: daypart = "night"

    return seg, daypart, top_branded_items(seg, daypart, n)["item_label"].tolist()

# test on a real store
sample_id = feature_table_final["STORE_ID"].iloc[0]
seg, daypart, recs = recommend_item_for_store(sample_id, hour_of_day=7)
print(f"Store {sample_id} -> {seg}, {daypart}: {recs}")

seg, daypart, recs = recommend_item_for_store(sample_id, hour_of_day=18)
print(f"Store {sample_id} -> {seg}, {daypart}: {recs}")

In [0]:
%pip install lets-plot

In [0]:
seg_demo = feature_table_final.groupby("final_segment")[["median_hh_income", "poverty_pct", "total_pop"]].mean().reset_index()
seg_demo_melt = seg_demo.melt(id_vars="final_segment", var_name="feature", value_name="value")

p_profile = (
    ggplot(seg_demo_melt, aes(x="final_segment", y="value", fill="final_segment"))
    + geom_bar(stat="identity")
    + facet_wrap("feature", scales="free_y")
    + labs(title="What Defines Each Store Segment", x="", y="")
    + theme_minimal() + theme(legend_position="none", axis_text_x=element_text(angle=45, hjust=1))
    + ggsize(850, 450)
)
p_profile.show()

In [0]:
from lets_plot import *
LetsPlot.setup_html()

top_item_per_cell = (
    segment_brand_profile.sort_values("units_sold", ascending=False)
    .groupby(["final_segment", "daypart"], as_index=False)
    .first()
)
top_item_per_cell["daypart"] = pd.Categorical(top_item_per_cell["daypart"], categories=["morning", "afternoon", "evening", "night"], ordered=True)
top_item_per_cell["item_short"] = top_item_per_cell["item_label"].str.split(" - ").str[0]  # brand name only, keeps labels short

color_map = {
    "Coca-Cola": "#2c7fb8",
    "Mountain Dew": "#d62728",
    "Red Bull": "#ff7f0e",
    "Monster": "#5c4033",   
    "BIC": "#9467bd",
    "Pepsi": "#2ca02c",
}

p_heatmap = (
    ggplot(top_item_per_cell, aes(x="daypart", y="final_segment", fill="item_short"))
    + geom_tile(width=0.95, height=0.95)
    + geom_text(aes(label="item_short"), color="white", size=8, fontface="bold")
    + scale_fill_manual(values=color_map)
    + labs(title="Top Recommended Product by Store Segment and Time of Day", x="", y="")
    + theme_minimal() + theme(legend_position="none")
    + ggsize(800, 450)
)
p_heatmap.show()

In [0]:
displayHTML(make_download_link(tree, "segment_tree.joblib"))

In [0]:
displayHTML(make_download_link(tree_feats, "tree_feats.joblib"))

In [0]:
displayHTML(make_download_link(feature_table_final[["STORE_ID"] + tree_feats + ["final_segment"]], "store_features.csv", is_dataframe=True))

In [0]:
segment_brand_profile_trimmed = (
    segment_brand_profile
    .sort_values("units_sold", ascending=False)
    .groupby(["final_segment", "daypart"])
    .head(10)
)

print(f"Full table: {len(segment_brand_profile)} rows")
print(f"Trimmed table: {len(segment_brand_profile_trimmed)} rows")

displayHTML(make_download_link(segment_brand_profile_trimmed, "segment_brand_profile.csv", is_dataframe=True))